 # Minimal example 
run the size mitigation planner on a handful of spws
and print the result.

Requires mitigation_table.py and smart_size_mitigation.py in the same
directory (or on PYTHONPATH).


In [8]:
from smart_size_mitigation_heuristics import build_mitigation_table, Limits, plan

# 1. Define the input parameters (example case)

In [9]:
spws = [
        {  # representative spw: nbin-eligible, single pointing, robust=2.0
            "spw": "25", "nchan": 3840, "ratio_satisfied": False,
            "pb": 0.9, "is_mosaic": False, "ppb": 4.0, "robust": 2.0,
            "n_roi": 3000, "n_total": 3840, "nfields": 5,
        },
        {  # smaller nchan, only eligible for nbin if ratio condition met
            "spw": "27", "nchan": 960, "ratio_satisfied": True,
            "pb": 1.0, "is_mosaic": False, "ppb": 3.5, "robust": None,
            "n_roi": 800, "n_total": 960, "nfields": 2,
        },
        {  # mosaic: PB mitigation forced to 0 regardless of pb value
            "spw": "29", "nchan": 7680, "ratio_satisfied": False,
            "pb": 0.8, "is_mosaic": False, "ppb": 4.0, "robust": None,
            "n_roi": 1000, "n_total": 7680, "nfields": 4,
        },
        {  # not nbin-eligible at all (ratio condition not met), tight PB already
            "spw": "31", "nchan": 1920, "ratio_satisfied": False,
            "pb": 0.65, "is_mosaic": False, "ppb": 5.0, "robust": 2.0,
            "n_roi": 1200, "n_total": 1920, "nfields": 3,
        }]

cube_sizes = {
    "25": 15.0, 
    "27": 15.0,
    "29": 70.0,
    "31": 15.0,
    }

product_sizes = {
    "25": 50.0, 
    "27": 50.0,
    "29": 150.0,
    "31": 50.0,
    }

# 2. Build the per-lever mitigation table

In [10]:
table = build_mitigation_table(spws, use_roi=False)
table

,n_bin,PB,cell,ROI,nfields_total,nfields,total_reduction
spw,,,,,,,
25,0.5,0.395062,0.339844,0.0,5,0.800000,0.960065
27,0.5,0.510000,0.265306,0.0,2,0.500000,0.910000
29,0.5,0.234375,0.437500,0.0,4,0.750000,0.946167
31,0.0,0.000000,0.577500,0.0,3,0.666667,0.859167


# 3. Set the size limits (GB)

In [15]:
limits = Limits(maxcubesize=10.0, maxproductsize=30.0, maxcubelimit=50.0)
limits


Limits(maxcubesize=10.0, maxproductsize=30.0, maxcubelimit=50.0)

# 4. Run the planner. spw "16" is the representative spw here

In [16]:
selected, msg = plan(
    table, 
    limits, 
    cube_sizes, 
    product_sizes, 
    repr_spw="29")

-- after Step 1 (per-spw cube mitigation) --
     min_steps    levers_used  achieved_reduction  mitigated_cube_GB  mitigated_product_GB
spw                                                                                       
25           1        nfields               0.800               3.00                10.000
27           1             PB               0.510               7.35                24.500
29           2  nfields,n_bin               0.875               8.75                18.750
31           1        nfields               0.667               5.00                16.667

-- Step 2b: global lever escalation -- 69.92 GB > budget 30.0 GB
     min_steps         levers_used  achieved_reduction  mitigated_cube_GB  mitigated_product_GB
spw                                                                                            
25           3  nfields,cell,n_bin               0.934              0.990                 3.301
27           3       PB,cell,n_bin               0.820 

# 5. Display the outcome.

In [17]:
print(f"\nResult: {msg['shortmsg']}")
if selected is not None:
    print(f"Spws kept: {list(selected.index)}")


Result: Size was mitigated
Spws kept: ['29', '25', '31', '27']


In [18]:
selected

,n_bin,PB,cell,ROI,nfields_total,nfields,total_reduction,cube_size_GB,product_size_GB,required_reduction,min_steps,levers_used,achieved_reduction,mitigated_cube_GB,mitigated_product_GB
spw,,,,,,,,,,,,,,,
29,0.5,0.234375,0.437500,0.0,4,0.750000,0.946167,70.0,150.0,0.857143,3,"nfields,n_bin,cell",0.929688,4.921875,10.546875
25,0.5,0.395062,0.339844,0.0,5,0.800000,0.960065,15.0,50.0,0.333333,3,"nfields,cell,n_bin",0.933984,0.990234,3.300781
31,0.0,0.000000,0.577500,0.0,3,0.666667,0.859167,15.0,50.0,0.333333,2,"nfields,cell",0.859167,2.112500,7.041667
27,0.5,0.510000,0.265306,0.0,2,0.500000,0.910000,15.0,50.0,0.333333,3,"PB,cell,n_bin",0.820000,2.700000,9.000000
